# Getting Features

This notebook demonstrates the creation of quantum circuits and their contraction plans. Once you acquire a plan, certain features will be figured out.
---


## Step 0: Loading software

 First of all, we create a new project to load all the neccesary software

In [2]:
] activate New_Project_on_QXTools;

  Activating new project at `C:\Users\alfre\Desktop\Julia\Setembre_2026\Github\18_features_notebook_1\New_Project_on_QXTools`


In [1]:
import Pkg; 
Pkg.add("QXTools")
Pkg.add("QXGraphDecompositions")
Pkg.add("QXZoo")
Pkg.add("DataStructures")
Pkg.add("QXTns")
Pkg.add("NDTensors")
Pkg.add("ITensors")
Pkg.add("LightGraphs")
Pkg.add("PyCall")




   Resolving package versions...
    Updating `C:\Users\Usuario\.julia\environments\v1.9\Project.toml`
  [84f0eee1] + QXTools v1.0.0
    Updating `C:\Users\Usuario\.julia\environments\v1.9\Manifest.toml`
  [84f0eee1] + QXTools v1.0.0
⌅ [6aa20fa7] + TensorOperations v3.2.5
        Info Packages marked with ⌅ have new versions available but compatibility constraints restrict them from upgrading. To see why use `status --outdated -m`
Precompiling project...
  ✓ GeometryBasics
  ✓ QXTools
  ✓ NetworkLayout
  ✓ GraphRecipes
  4 dependencies successfully precompiled in 46 seconds. 263 already precompiled. 1 skipped during auto due to previous errors.
   Resolving package versions...
  No Changes to `C:\Users\Usuario\.julia\environments\v1.9\Project.toml`
  No Changes to `C:\Users\Usuario\.julia\environments\v1.9\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\Usuario\.julia\environments\v1.9\Project.toml`
  No Changes to `C:\Users\Usuario\.julia\environments\v1.9\Ma

In [2]:
using QXTools
using QXTns
using QXZoo
using PyCall
using QXGraphDecompositions
using LightGraphs
using DataStructures
using TimerOutputs
using ITensors
using LinearAlgebra
using NDTensors
using CUDA

[ Info: OMEinsum loaded the CUDA module successfully


In [3]:
# Load custom functions
include("../src/funcions_article_IA.jl");

## Step 1: Create a GHZ Circuit
We begin by creating a GHZ circuit based on the user-defined number of qubits.

In [ ]:
 begin
        # --- Step 1: GHZ Creation ---
       @info("How many qubits do you want for the GHZ circuit(n)?\n\n")

                     N = readline()
                     n = parse(Int, N)


               # Create GHZ circuit
              cct = create_ghz_circuit(n)

              @info(" circuit GHZ with  $(n) qubits created\n\n")

        tnc = convert_to_tnc(cct)  # Convert the GHZ circuit into a tensor network circuit
            end


---

## Step 2: Contraction plans
We perform contraction plans of the tensor network using the Flow cutter algorithm. We can use Girvan–Newman algorithm and others to get a contraction order.

In [ ]:
# Find a good contraction plan
plan = flow_cutter_contraction_plan(tnc; time=10)

---
## Step 3: Getting contraction features in a file
We perform  a fake contraction in the GPU to get some tensor features.

In [ ]:

using CUDA
begin
        for i in keys(tnc.tn.:tensor_map)
                             tnc.tn.:tensor_map[i].storage= CuArray(tnc.tn.:tensor_map[i].storage)
                         end

        Dades= contract_tn_rank_mock!(tnc.tn, plan,mock= true,verbose=true)

 end


A file named ``fitxer_sortida.txt`` is generated .

In [ ]:
readlines("fitxer_sortida.txt")

### Brief explanation 

- **t1, t2, t3**: names of the tensors (first three words)  
  These are the identifiers of the two input tensors being contracted and the resulting output tensor. For example, in a line like `t518 t564 I1 ...`, `t518` and `t564` are the two tensors being contracted, and `I1` is the tensor that results from the contraction.

- **r1, r2, r3**: ranks (or orders) of the tensors (words 4, 5, 6)  
  These are integer values indicating the number of indices (dimensions) of each tensor. For example, a rank of `2` means the tensor has 2 indices, a rank of `1` means it is a vector-like tensor, and so on. The three values correspond respectively to the ranks of `t1`, `t2`, and `t3`.

- **indices**: common contraction indices (word 7)  
  This field describes the shared indices between the two tensors being contracted. It specifies which index labels (e.g., `id=186`, `"Qubit16"`, with possible prime markers like `'19`) are summed over during the contraction. In tensor network language, these are the "bond" indices that connect the two tensors.

- **cost**: contraction time (last word)  
  This is a floating-point number representing the computational cost or execution time associated with performing this particular contraction step. It is typically used to evaluate or compare the efficiency of a contraction plan.

  
  *Note*: In this example, we do not contract "really" in a GPU, so the time results are negligible

---

### Summary
This notebook demonstrated:
1. The creation of a GHZ tensor network.
2. Contraction plan using the FlowCutter algorithm.
3. Convert this plan into a file with some tensor characteristics to get future features


Thank you for exploring tensor network construction with us!